In [4]:
import pandas as pd  
import numpy as np  
import os  
import re  
import glob  
import io
import calendar
from datetime import datetime  
import msoffcrypto
import warnings
from dateutil.relativedelta import relativedelta

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

In [5]:
import calendar  
import glob  
import io  
import os  
import re

import msoffcrypto  
import pandas as pd  
from msoffcrypto.exceptions import FileFormatError, InvalidKeyError


def read_protected_excel(file_path, password, sheet_names, skiprows_list):  
    """Read multiple sheets from an Excel file that may or may not be password-protected."""  
    with open(file_path, 'rb') as f:  
        raw = io.BytesIO(f.read())

    workbook = raw  
    encrypted = False

    try:  
        raw.seek(0)  
        office_file = msoffcrypto.OfficeFile(raw)  
        if office_file.is_encrypted():  
            encrypted = True  
            decrypted = io.BytesIO()  
            office_file.load_key(password=password)  
            office_file.decrypt(decrypted)  
            workbook = decrypted  
    except FileFormatError:  
        workbook = raw  
    except InvalidKeyError:  
        raise RuntimeError(f"Wrong password for {os.path.basename(file_path)}")

    print(f"    ({'encrypted' if encrypted else 'not encrypted'})")

    dfs = []  
    for sheet_name, skiprows in zip(sheet_names, skiprows_list):  
        workbook.seek(0)  
        df = pd.read_excel(workbook, sheet_name=sheet_name, skiprows=skiprows)  
        dfs.append(df)

    return dfs


def clean_columns(df):  
    df.columns = (  
        df.columns.astype(str)  
        .str.strip()  
        .str.replace(r'[^a-zA-Z0-9]', '_', regex=True)  
        .str.replace(r'_+', '_', regex=True)  
        .str.strip('_')  
        .str.lower()  
    )  
    return df


# ---------------- config ----------------  
base_paths = [  
    'L:/2023 Pilar Plant Files/2023 Manpower Detail/',  
    'L:/2024 Pilar Plant Files/2024 Manpower Detail/',  
    'L:/2025 Pilar Plant Files/2025 Manpower Detail/',  
    'L:/2026 Pilar Plant Files/2026 Manpower Detail/',  
]  
password = 'bamhorse2'  
sheet_names = ['MTD FTEs', 'MTD Dollars']  
skiprows_list = [7, 7]  
required_names = ['LENOX SAL']  
start_date = pd.to_datetime('2023-01-01')

# ---------------- collect files across ALL paths ----------------  
all_files = []  
for base_path in base_paths:  
    if not os.path.isdir(base_path):  
        print(f"WARNING: path not found, skipping -> {base_path}")  
        continue  
    found = glob.glob(os.path.join(base_path, '*.xlsm'))  
    print(f"{len(found):4d} .xlsm files in {base_path}")  
    all_files.extend(found)

# ---------------- filter ----------------  
month_end_files = []  
seen = set()

for f in all_files:  
    filename = os.path.basename(f)

    if filename.startswith('~$'):          # Excel lock/temp file  
        continue

    upper_name = filename.upper()

    matched_name = next(  
        (name for name in required_names if name.upper() in upper_name), None  
    )  
    if matched_name is None:  
        continue

    site_code = matched_name.split(' ')[0]

    date_match = re.search(r'(\d{4}-\d{2}-\d{2})', filename)  
    if not date_match:  
        print(f"  WARNING: Could not parse date from {filename}")  
        continue

    date_str = date_match.group(1)  
    file_date = pd.to_datetime(date_str, errors='coerce')  
    if pd.isna(file_date):  
        print(f"  WARNING: Bad date '{date_str}' in {filename}")  
        continue

    if file_date < start_date:  
        continue

    last_day = calendar.monthrange(file_date.year, file_date.month)[1]  
    if file_date.day != last_day:  
        continue

    key = (site_code, date_str)            # dedupe same site+month across folders  
    if key in seen:  
        print(f"  DUPLICATE skipped: {filename}")  
        continue  
    seen.add(key)

    month_end_files.append((f, date_str, site_code))

month_end_files.sort(key=lambda x: (x[1], x[2]))

print(f"\nMonth-end files found (on/after {start_date.date()}): {len(month_end_files)}")  
for f, d, name in month_end_files:  
    print(f"  {os.path.basename(f)} (date: {d} | site: {name})")

if not month_end_files:  
    raise SystemExit("No month-end files matched. Check base_paths / required_names / start_date.")

# ---------------- read ----------------  
files = [(f, 'Monthly', d, name) for f, d, name in month_end_files]

all_mtd_ftes = []  
all_mtd_doll = []  
failed = []

for file_path, pay_type, date, site in files:  
    print(f"\nReading {pay_type} | Site: {site} | Date: {date}")  
    try:  
        dfs = read_protected_excel(file_path, password, sheet_names, skiprows_list)  
        for df in dfs:  
            df['Pay_Type'] = pay_type  
            df['Report_Date'] = pd.to_datetime(date)  
            df['BU'] = site  
            df['Source_File'] = os.path.basename(file_path)

        all_mtd_ftes.append(clean_columns(dfs[0]))  
        all_mtd_doll.append(clean_columns(dfs[1]))

    except Exception as e:  
        print(f"  ERROR: {type(e).__name__}: {e}")  
        failed.append((os.path.basename(file_path), str(e)))

if not all_mtd_ftes:  
    raise SystemExit("Every file failed to read. See errors above.")

mtd_ftes_combined = pd.concat(all_mtd_ftes, ignore_index=True)  
mtd_doll_combined = pd.concat(all_mtd_doll, ignore_index=True)

# ---------------- summary ----------------  
print("\n--- Summary ---")  
print(f"Files attempted: {len(files)} | succeeded: {len(all_mtd_ftes)} | failed: {len(failed)}")  
if failed:  
    for name, err in failed:  
        print(f"  FAILED: {name} -> {err}")  
print(f"MTD FTEs:    {mtd_ftes_combined.shape}")  
print(f"MTD Dollars: {mtd_doll_combined.shape}")  
print(f"Sites found: {sorted(mtd_doll_combined['bu'].unique())}")  
print(f"Date range:  {mtd_doll_combined['report_date'].min().date()} "  
      f"to {mtd_doll_combined['report_date'].max().date()}")  

 178 .xlsm files in L:/2023 Pilar Plant Files/2023 Manpower Detail/
 152 .xlsm files in L:/2024 Pilar Plant Files/2024 Manpower Detail/
 148 .xlsm files in L:/2025 Pilar Plant Files/2025 Manpower Detail/
 137 .xlsm files in L:/2026 Pilar Plant Files/2026 Manpower Detail/

Month-end files found (on/after 2023-01-01): 44
  LENOX SALAERN LENOX_EXEC 2023-01-31.xlsm (date: 2023-01-31 | site: LENOX)
  LENOX SALAERN LENOX_EXEC 2023-02-28.xlsm (date: 2023-02-28 | site: LENOX)
  LENOX SALAERN LENOX_EXEC 2023-03-31.xlsm (date: 2023-03-31 | site: LENOX)
  LENOX SALAERN LENOX_EXEC 2023-04-30.xlsm (date: 2023-04-30 | site: LENOX)
  LENOX SALAERN LENOX_EXEC 2023-05-31.xlsm (date: 2023-05-31 | site: LENOX)
  LENOX SALAERN LENOX_EXEC 2023-06-30.xlsm (date: 2023-06-30 | site: LENOX)
  LENOX SALAERN LENOX_EXEC 2023-07-31.xlsm (date: 2023-07-31 | site: LENOX)
  LENOX SALAERN LENOX_EXEC 2023-08-31.xlsm (date: 2023-08-31 | site: LENOX)
  LENOX SALAERN LENOX_EXEC 2023-09-30.xlsm (date: 2023-09-30 | site: LE

In [6]:
display(mtd_ftes_combined)

,deptid,department_description,job_code,job_code_description,regular,overtime,holiday_wkd,non_prod,total,regular_non_prod,...,holiday_wkd,total,regular_non_prod,overtime,holiday_wkd,total,pay_type,report_date,bu,source_file
0,15600000,Hospital Administration,107015,Site CMIO,0.86,0.00,0.0,0.14,1.00,1.00,...,0.0,1.00,0.00,0.00,0.0,0.00,Monthly,2023-01-31,LENOX,LENOX SALAERN LENOX_EXEC 2023-01-31.xlsm
1,NaN,NaN,200222,"AVP, IT&S Strategic Planning",0.83,0.00,0.0,0.12,0.95,0.90,...,0.0,0.90,-0.05,0.00,0.0,-0.05,Monthly,2023-01-31,LENOX,LENOX SALAERN LENOX_EXEC 2023-01-31.xlsm
2,NaN,NaN,200274,Clinical Engineering Technician,0.00,0.02,0.0,0.00,0.02,0.00,...,0.0,0.00,0.00,-0.02,0.0,-0.02,Monthly,2023-01-31,LENOX,LENOX SALAERN LENOX_EXEC 2023-01-31.xlsm
3,NaN,NaN,200876,IT&S Project Manager,0.00,0.00,0.0,0.00,0.00,1.00,...,0.0,1.00,1.00,0.00,0.0,1.00,Monthly,2023-01-31,LENOX,LENOX SALAERN LENOX_EXEC 2023-01-31.xlsm
4,NaN,NaN,201889,"Supervisor, Administrative Support",0.64,0.00,0.0,0.36,1.00,1.00,...,0.0,1.00,0.00,0.00,0.0,0.00,Monthly,2023-01-31,LENOX,LENOX SALAERN LENOX_EXEC 2023-01-31.xlsm
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55640,NaN,NaN,108626,ER Expeditor,0.00,0.00,0.0,0.00,0.00,0.00,...,0.0,0.00,0.00,0.00,0.0,0.00,Monthly,2026-08-31,LENOX,LENOX SALAERN LENOX_EXEC 2026-08-31.xlsm
55641,15699000 Total,NaN,NaN,NaN,0.31,0.00,0.0,0.19,0.50,0.00,...,0.0,0.00,-0.50,0.00,0.0,-0.50,Monthly,2026-08-31,LENOX,LENOX SALAERN LENOX_EXEC 2026-08-31.xlsm
55642,(blank),(blank),(blank),(blank),NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Monthly,2026-08-31,LENOX,LENOX SALAERN LENOX_EXEC 2026-08-31.xlsm
55643,(blank) Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Monthly,2026-08-31,LENOX,LENOX SALAERN LENOX_EXEC 2026-08-31.xlsm


In [7]:
# function to clean tabs
def clean_sheet(sheet):
    # remove subtotal rowa
    sheet_no_totals = sheet[~sheet['deptid'].str.contains('total', case=False, na=False)]

    # forward fill department ids and names
    sheet_no_totals['deptid'] = sheet_no_totals['deptid'].ffill()
    sheet_no_totals['department_description'] = sheet_no_totals['department_description'].ffill()

    # remove department ids that are blank
    sheet_no_nulls = sheet_no_totals[sheet_no_totals['deptid'] != '(blank)']
    # set type to integer
    sheet_no_nulls[['deptid', 'job_code']] = sheet_no_nulls[['deptid', 'job_code']].astype('Int64') 

    # fix duplicate column names  
    cols = pd.Series(sheet_no_nulls.columns)  
    for dup in cols[cols.duplicated()].unique():  
        count = 0  
        for i in range(len(cols)):  
            if cols[i] == dup:  
                count += 1  
                if count > 1:  
                    cols[i] = f"{dup}_{count}"  
    sheet_no_nulls.columns = cols

    sheet_filtered = sheet_no_nulls[['report_date', 'bu', 'deptid', 'department_description', 'job_code', 'job_code_description',  
                                     'overtime', 'total', 'overtime_2', 'total_2']]

    sheet_calced = sheet_filtered.copy()
    sheet_calced['reg_a'] = sheet_calced['total'] - sheet_calced['overtime']
    sheet_calced['ot_a'] = sheet_calced['overtime']
    sheet_calced['total_a'] = sheet_calced['total']

    sheet_calced['reg_b'] = sheet_calced['total_2'] - sheet_calced['overtime_2']
    sheet_calced['ot_b'] = sheet_calced['overtime_2']
    sheet_calced['total_b'] = sheet_calced['total_2']

    sheet_final = sheet_calced.drop(columns=['overtime', 'total', 'overtime_2', 'total_2'])

    return sheet_final

ftes_all = clean_sheet(mtd_ftes_combined)
dollars_all = clean_sheet(mtd_doll_combined)

display(ftes_all)

,report_date,bu,deptid,department_description,job_code,job_code_description,reg_a,ot_a,total_a,reg_b,ot_b,total_b
0,2023-01-31,LENOX,15600000,Hospital Administration,107015,Site CMIO,1.00,0.00,1.00,1.0,0.0,1.0
1,2023-01-31,LENOX,15600000,Hospital Administration,200222,"AVP, IT&S Strategic Planning",0.95,0.00,0.95,0.9,0.0,0.9
2,2023-01-31,LENOX,15600000,Hospital Administration,200274,Clinical Engineering Technician,0.00,0.02,0.02,0.0,0.0,0.0
3,2023-01-31,LENOX,15600000,Hospital Administration,200876,IT&S Project Manager,0.00,0.00,0.00,1.0,0.0,1.0
4,2023-01-31,LENOX,15600000,Hospital Administration,201889,"Supervisor, Administrative Support",1.00,0.00,1.00,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
55636,2026-08-31,LENOX,15699000,Corporate Retained Expenses,203068,Senior GME Training Program Administrator,0.00,0.00,0.00,0.0,0.0,0.0
55637,2026-08-31,LENOX,15699000,Corporate Retained Expenses,101024,"Chief Resident (LJ,MN)",0.00,0.00,0.00,0.0,0.0,0.0
55638,2026-08-31,LENOX,15699000,Corporate Retained Expenses,103331,"Chief Fellow (LJ,MN)",0.00,0.00,0.00,0.0,0.0,0.0
55639,2026-08-31,LENOX,15699000,Corporate Retained Expenses,903901,Pharmacy Resident,0.00,0.00,0.00,0.0,0.0,0.0


In [18]:
vp_xwalk = pd.read_excel(  
    'L:/2026 Pilar Plant Files/Crosswalks/Manhattan Region Crosswalk.xlsx',  
    sheet_name='vp_dept'  
)

def spec_vp(sheet, vp=None):  
    sheet_vps = pd.merge(vp_xwalk, sheet, how='right', left_on='Dept', right_on='deptid').drop(columns=['Dept']).rename(columns={'VP': 'vp'})

    if vp is None:  
        return sheet_vps

    if vp == 'Browne, Kevin':  
        kevin_gv_depts = [74002506, 74002520, 74002561]  
        sheet_spec = sheet_vps[(sheet_vps['vp'] == vp) | (sheet_vps['deptid'].isin(kevin_gv_depts))]  
        sheet_kevin = sheet_spec[sheet_spec['deptid'] != 15600090]  
        return sheet_kevin  
    else:  
        sheet_spec = sheet_vps[sheet_vps['vp'] == vp]  
        return sheet_spec  

ftes = spec_vp(ftes_all)
#dollars = spec_vp(dollars_all)
ftes['flex_flag'] = ftes['job_code_description'].str.lower().str.contains('flexstaff', na=False)

ftes_summary = ftes.groupby(['report_date', 'flex_flag'])[['reg_a', 'ot_a', 'total_a']].sum().reset_index()

file_name = 'fte_summary'

with pd.ExcelWriter(f'{file_name}.xlsx', engine='openpyxl') as writer: 
    ftes_summary.to_excel(writer, sheet_name='FTE Summary', index=False)  
    ftes.to_excel(writer, sheet_name='MTD FTEs', index=False)  
    #dollars.to_excel(writer, sheet_name='MTD Dollars', index=False)

print(f"Written to {file_name}.xlsx")

Written to fte_summary.xlsx
